# Анализ средних рангов аннотаторов
Построение усредненных рангов резюме по оценкам двух аннотаторов.

Ноутбук выводит файл `averaged_ranks_analysis.xlsx` с 5 листами:
1. **Средние ранги** — матрица 30×5
2. **Данные для модели** — long format (150 строк)
3. **Ранжирование по вакансиям** — отсортированные резюме для каждой вакансии
4. **Распределение рангов** — гистограмма 9 средних рангов
5. **Комбинации аннотаторов** — кросс-таблица частот


## 0. Установка зависимостей и импорт


In [49]:
!pip install openpyxl -q

import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter


## 1. Исходные данные аннотаторов

Матрицы рангов: строки — резюме (CV 1–30), столбцы — вакансии (V1–V5).
Значение ячейки — ранг от 1 (лучшее соответствие) до 5 (худшее).


In [50]:
# ---------- Матрица Аннотатор 1 ----------
data_ann1 = [
    [2,1,4,3,5],[1,2,3,4,5],[1,2,3,4,5],[3,1,2,4,5],[1,5,4,2,3],
    [3,2,1,4,5],[3,2,1,5,4],[2,4,3,1,5],[1,5,2,1,4],[3,2,1,4,5],
    [1,2,3,4,5],[1,2,3,4,5],[1,3,2,4,5],[1,2,3,4,5],[3,1,2,4,5],
    [3,1,2,4,5],[3,1,2,4,5],[1,2,5,3,4],[3,2,1,4,5],[3,2,1,4,5],
    [2,3,1,4,5],[1,2,3,5,4],[2,1,3,5,4],[1,2,3,5,4],[1,2,3,4,5],
    [2,1,3,4,5],[2,3,4,5,1],[2,4,3,2,5],[5,1,2,4,3],[2,1,4,3,5],
]

# ---------- Матрица Аннотатор 2 ----------
data_ann2 = [
    [4,3,1,5,2],[2,4,3,1,5],[5,4,2,3,1],[1,3,2,4,5],[5,1,2,4,3],
    [1,3,2,4,5],[4,2,3,1,5],[2,4,3,1,5],[3,4,2,1,5],[4,1,2,5,3],
    [2,4,3,5,1],[4,3,2,1,5],[4,2,3,1,5],[3,4,2,1,5],[2,4,3,1,5],
    [3,2,4,1,5],[4,2,3,1,5],[4,2,5,3,1],[4,2,3,1,5],[1,5,2,4,3],
    [1,3,4,5,2],[4,1,3,2,5],[1,3,4,2,5],[1,4,3,5,2],[1,4,2,5,3],
    [1,5,2,4,3],[4,3,1,2,5],[1,4,2,3,5],[5,1,2,4,3],[1,2,3,4,5],
]

vacancies = ["V1", "V2", "V3", "V4", "V5"]
cv_labels = [f"CV {i}" for i in range(1, 31)]

ann1 = pd.DataFrame(data_ann1, columns=vacancies, index=cv_labels)
ann2 = pd.DataFrame(data_ann2, columns=vacancies, index=cv_labels)

print("Матрица Аннотатор 1 (первые 5 строк):")
display(ann1.head())
print("\nМатрица Аннотатор 2 (первые 5 строк):")
display(ann2.head())


Матрица Аннотатор 1 (первые 5 строк):


,V1,V2,V3,V4,V5
CV 1,2,1,4,3,5
CV 2,1,2,3,4,5
CV 3,1,2,3,4,5
CV 4,3,1,2,4,5
CV 5,1,5,4,2,3



Матрица Аннотатор 2 (первые 5 строк):


,V1,V2,V3,V4,V5
CV 1,4,3,1,5,2
CV 2,2,4,3,1,5
CV 3,5,4,2,3,1
CV 4,1,3,2,4,5
CV 5,5,1,2,4,3


## 2. Вычисление средних рангов

Для каждой пары (резюме, вакансия) средний ранг = (Ранг_А1 + Ранг_А2) / 2.


In [51]:
avg_ranks = (ann1 + ann2) / 2

print("Матрица средних рангов (первые 10 строк):")
display(avg_ranks.head(10))

unique_ranks = sorted(avg_ranks.values.flatten().tolist())
unique_set = sorted(set(unique_ranks))
print(f"\nУникальные средние ранги ({len(unique_set)} шт.): {unique_set}")


Матрица средних рангов (первые 10 строк):


,V1,V2,V3,V4,V5
CV 1,3.0,2.0,2.5,4.0,3.5
CV 2,1.5,3.0,3.0,2.5,5.0
CV 3,3.0,3.0,2.5,3.5,3.0
CV 4,2.0,2.0,2.0,4.0,5.0
CV 5,3.0,3.0,3.0,3.0,3.0
CV 6,2.0,2.5,1.5,4.0,5.0
CV 7,3.5,2.0,2.0,3.0,4.5
CV 8,2.0,4.0,3.0,1.0,5.0
CV 9,2.0,4.5,2.0,1.0,4.5
CV 10,3.5,1.5,1.5,4.5,4.0



Уникальные средние ранги (9 шт.): [1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]


## 3. Стили для Excel

Цветовая палитра: зелёный (ранг 1.0 — лучший) → серый (ранг 5.0 — худший).


In [52]:
# Палитра цветов для средних рангов
RANK_COLORS = {
    1.0: "C6EFCE", 1.5: "D5F5E3", 2.0: "FFF2CC", 2.5: "FFE0B2",
    3.0: "FFCCBC", 3.5: "F8BBD0", 4.0: "E1BEE7", 4.5: "D5D5D5",
    5.0: "BDBDBD",
}

# Общие стили
HEADER_FONT  = Font(name="Arial", bold=True, size=11, color="FFFFFF")
HEADER_FILL  = PatternFill("solid", fgColor="4472C4")
NORMAL_FONT  = Font(name="Arial", size=10)
BOLD_FONT    = Font(name="Arial", size=10, bold=True)
TITLE_FONT   = Font(name="Arial", bold=True, size=14)
CENTER       = Alignment(horizontal="center", vertical="center")
THIN_BORDER  = Border(
    left=Side(style="thin"), right=Side(style="thin"),
    top=Side(style="thin"), bottom=Side(style="thin"),
)

def style_cell(cell, font=NORMAL_FONT, fill=None, align=CENTER, border=THIN_BORDER):
    """Применяет стили к ячейке."""
    cell.font = font
    cell.alignment = align
    cell.border = border
    if fill:
        cell.fill = PatternFill("solid", fgColor=fill)

def rank_fill(val):
    """Возвращает цвет заливки по среднему рангу."""
    return RANK_COLORS.get(val, "FFFFFF")

print("Стили готовы")


Стили готовы


## 4. Создание книги Excel


In [53]:
wb = Workbook()
print("Пустая книга создана")


Пустая книга создана


### 4.1 Лист «Средние ранги»


In [54]:
ws1 = wb.active
ws1.title = "Средние ранги"

# Заголовок
ws1.merge_cells("A1:F1")
ws1["A1"] = "Средние ранги по вакансиям (Аннотатор 1 + Аннотатор 2) / 2"
ws1["A1"].font = TITLE_FONT
ws1["A1"].alignment = Alignment(horizontal="center")

# Шапка таблицы
for col, header in enumerate(["Резюме"] + vacancies, 1):
    cell = ws1.cell(row=3, column=col, value=header)
    style_cell(cell, font=HEADER_FONT, fill="4472C4")

# Данные
for i, (cv, row) in enumerate(avg_ranks.iterrows(), 4):
    c = ws1.cell(row=i, column=1, value=cv)
    style_cell(c, font=BOLD_FONT)
    for j, vac in enumerate(vacancies, 2):
        val = row[vac]
        cell = ws1.cell(row=i, column=j, value=val)
        style_cell(cell, fill=rank_fill(val))

ws1.column_dimensions["A"].width = 12
for col in range(2, 7):
    ws1.column_dimensions[get_column_letter(col)].width = 10

print("Лист «Средние ранги» — готов")


Лист «Средние ранги» — готов


### 4.2 Лист «Данные для модели»


In [55]:
ws2 = wb.create_sheet("Данные для модели")

ws2.merge_cells("A1:F1")
ws2["A1"] = "Данные для обучения модели (long format)"
ws2["A1"].font = TITLE_FONT
ws2["A1"].alignment = Alignment(horizontal="center")

headers_ml = ["Вакансия", "Резюме", "Ранг А1", "Ранг А2", "Средний ранг", "Комбинация"]
for col, h in enumerate(headers_ml, 1):
    style_cell(ws2.cell(row=3, column=col, value=h), font=HEADER_FONT, fill="4472C4")

row_num = 4
for vac in vacancies:
    for cv in cv_labels:
        r1 = int(ann1.loc[cv, vac])
        r2 = int(ann2.loc[cv, vac])
        avg_val = avg_ranks.loc[cv, vac]

        vac_num = int(vac.replace("V", ""))
        cv_num  = int(cv.replace("CV ", ""))

        ws2.cell(row=row_num, column=1, value=vac_num)
        ws2.cell(row=row_num, column=2, value=cv_num)
        ws2.cell(row=row_num, column=3, value=r1)
        ws2.cell(row=row_num, column=4, value=r2)
        cell5 = ws2.cell(row=row_num, column=5, value=avg_val)
        style_cell(cell5, fill=rank_fill(avg_val))
        ws2.cell(row=row_num, column=6, value=f"{r1}-{r2}")

        for c in [1, 2, 3, 4, 6]:
            cell = ws2.cell(row=row_num, column=c)
            cell.font = NORMAL_FONT
            cell.alignment = CENTER
            cell.border = THIN_BORDER

        row_num += 1

for col in range(1, 7):
    ws2.column_dimensions[get_column_letter(col)].width = 14

print(f"Лист «Данные для модели» — {row_num - 4} строк")


Лист «Данные для модели» — 150 строк


### 4.3 Лист «Ранжирование по вакансиям»


In [56]:
ws3 = wb.create_sheet("Ранжирование по вакансиям")

start_row = 1
for vac in vacancies:
    vac_label = f'Вакансия {vac.replace("V", "")}'

    # Заголовок вакансии
    ws3.merge_cells(start_row=start_row, start_column=1, end_row=start_row, end_column=5)
    ws3.cell(row=start_row, column=1, value=vac_label).font = Font(
        name="Arial", bold=True, size=13, color="4472C4"
    )

    # Шапка
    start_row += 1
    for col, h in enumerate(["№", "Резюме", "Ранг А1", "Ранг А2", "Средний ранг"], 1):
        style_cell(ws3.cell(row=start_row, column=col, value=h), font=HEADER_FONT, fill="4472C4")

    # Сортировка резюме по среднему рангу
    sorted_cvs = avg_ranks[vac].sort_values()

    for pos, (cv, avg_val) in enumerate(sorted_cvs.items(), 1):
        r = start_row + pos
        r1 = int(ann1.loc[cv, vac])
        r2 = int(ann2.loc[cv, vac])

        ws3.cell(row=r, column=1, value=pos)
        ws3.cell(row=r, column=2, value=cv)
        ws3.cell(row=r, column=3, value=r1)
        ws3.cell(row=r, column=4, value=r2)
        cell5 = ws3.cell(row=r, column=5, value=avg_val)
        cell5.fill = PatternFill("solid", fgColor=rank_fill(avg_val))

        for c in range(1, 6):
            ws3.cell(row=r, column=c).font = NORMAL_FONT
            ws3.cell(row=r, column=c).alignment = CENTER
            ws3.cell(row=r, column=c).border = THIN_BORDER

    start_row += 30 + 2  # 30 CV + шапка + пустая строка

for col in range(1, 6):
    ws3.column_dimensions[get_column_letter(col)].width = 14

print("Лист «Ранжирование по вакансиям» — готов")


Лист «Ранжирование по вакансиям» — готов


### 4.4 Лист «Распределение рангов»


In [57]:
ws4 = wb.create_sheet("Распределение рангов")

ws4.merge_cells("A1:C1")
ws4["A1"] = "Распределение средних рангов"
ws4["A1"].font = TITLE_FONT
ws4["A1"].alignment = Alignment(horizontal="center")

for col, h in enumerate(["Средний ранг", "Кол-во записей", "Доля (%)"], 1):
    style_cell(ws4.cell(row=3, column=col, value=h), font=HEADER_FONT, fill="4472C4")

all_avg = avg_ranks.values.flatten()
total = len(all_avg)
unique_ranks_sorted = sorted(set(all_avg))

for i, rank in enumerate(unique_ranks_sorted, 4):
    count = int(np.sum(all_avg == rank))
    pct = round(count / total * 100, 1)

    ws4.cell(row=i, column=1, value=rank)
    ws4.cell(row=i, column=2, value=count)
    ws4.cell(row=i, column=3, value=pct)

    for c in range(1, 4):
        cell = ws4.cell(row=i, column=c)
        cell.font = NORMAL_FONT
        cell.alignment = CENTER
        cell.border = THIN_BORDER
    ws4.cell(row=i, column=1).fill = PatternFill("solid", fgColor=rank_fill(rank))

# Строка итого
row_total = 4 + len(unique_ranks_sorted)
ws4.cell(row=row_total, column=1, value="ИТОГО").font = BOLD_FONT
ws4.cell(row=row_total, column=2, value=total)
ws4.cell(row=row_total, column=3, value=100.0)

# Пояснения
info_row = row_total + 2
notes = [
    ("Система рангов:", True),
    ("Исходная шкала аннотаторов: 5 рангов (1-5)", False),
    ("Средняя шкала: 9 рангов (1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0)", False),
    ("Возможных комбинаций (A1, A2): 1-1, 1-2, 1-3, 1-4, 1-5, 2-2, 2-3, ..., 5-5", False),
    ("Для округления до 5 рангов: 1.0→1, 1.5→2, 2.0→2, 2.5→3, 3.0→3, 3.5→4, 4.0→4, 4.5→5, 5.0→5", False),
]
for j, (text, bold) in enumerate(notes):
    cell = ws4.cell(row=info_row + j, column=1, value=text)
    cell.font = Font(name="Arial", bold=bold, size=10 + (bold * 1))

for col in range(1, 4):
    ws4.column_dimensions[get_column_letter(col)].width = 18

print("Лист «Распределение рангов» — готов")

# Вывод таблицы распределения в ноутбуке
dist_df = pd.DataFrame({
    "Средний ранг": unique_ranks_sorted,
    "Кол-во": [int(np.sum(all_avg == r)) for r in unique_ranks_sorted],
    "Доля (%)": [round(np.sum(all_avg == r) / total * 100, 1) for r in unique_ranks_sorted],
})
display(dist_df)


Лист «Распределение рангов» — готов


,Средний ранг,Кол-во,Доля (%)
0,1.0,5,3.3
1,1.5,15,10.0
2,2.0,17,11.3
3,2.5,31,20.7
4,3.0,31,20.7
5,3.5,14,9.3
6,4.0,12,8.0
7,4.5,9,6.0
8,5.0,16,10.7


### 4.5 Лист «Комбинации аннотаторов»


In [58]:
ws5 = wb.create_sheet("Комбинации аннотаторов")

ws5.merge_cells("A1:G1")
ws5["A1"] = "Частотность комбинаций рангов аннотаторов"
ws5["A1"].font = TITLE_FONT
ws5["A1"].alignment = Alignment(horizontal="center")

# Подсчёт комбинаций
combos = {}
for vac in vacancies:
    for cv in cv_labels:
        r1 = int(ann1.loc[cv, vac])
        r2 = int(ann2.loc[cv, vac])
        combos[(r1, r2)] = combos.get((r1, r2), 0) + 1

# Шапка
style_cell(ws5.cell(row=3, column=1, value="А1 \\ А2"), font=HEADER_FONT, fill="4472C4")
for j in range(1, 6):
    style_cell(ws5.cell(row=3, column=j + 1, value=f"А2={j}"), font=HEADER_FONT, fill="4472C4")
style_cell(ws5.cell(row=3, column=7, value="Итого"), font=HEADER_FONT, fill="4472C4")

# Данные
for i in range(1, 6):
    style_cell(
        ws5.cell(row=3 + i, column=1, value=f"А1={i}"),
        font=BOLD_FONT, fill="D9E2F3",
    )
    row_total_val = 0
    for j in range(1, 6):
        val = combos.get((i, j), 0)
        avg_r = (i + j) / 2
        cell = ws5.cell(row=3 + i, column=j + 1, value=val)
        style_cell(cell, fill=rank_fill(avg_r))
        row_total_val += val
    style_cell(ws5.cell(row=3 + i, column=7, value=row_total_val))

for col in range(1, 8):
    ws5.column_dimensions[get_column_letter(col)].width = 12

print("Лист «Комбинации аннотаторов» — готов")

# Вывод кросс-таблицы в ноутбуке
cross = pd.DataFrame(0, index=[f"А1={i}" for i in range(1,6)],
                         columns=[f"А2={j}" for j in range(1,6)])
for (r1, r2), cnt in combos.items():
    cross.loc[f"А1={r1}", f"А2={r2}"] = cnt
cross["Итого"] = cross.sum(axis=1)
display(cross)


Лист «Комбинации аннотаторов» — готов


,А2=1,А2=2,А2=3,А2=4,А2=5,Итого
А1=1,5,8,7,6,4,30
А1=2,7,7,6,10,1,31
А1=3,3,8,11,6,1,29
А1=4,11,2,2,7,8,30
А1=5,4,5,4,1,16,30


## 5. Сохранение файла


In [59]:
OUTPUT_PATH = "averaged_ranks_analysis.xlsx"
wb.save(OUTPUT_PATH)
print(f"\n Файл сохранен: {OUTPUT_PATH}")
print(f"   Листов: {len(wb.sheetnames)} — {wb.sheetnames}")

# Скачивание в Google Colab
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
    print("   Файл скачался")
except ImportError:
    print("   (Файл сохранен локально)")



 Файл сохранен: averaged_ranks_analysis.xlsx
   Листов: 5 — ['Средние ранги', 'Данные для модели', 'Ранжирование по вакансиям', 'Распределение рангов', 'Комбинации аннотаторов']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   Файл скачался
